# 1. Config 파일 읽기, WandB 연결, 라이브러리 import

In [ ]:
# config 파일 불러오기
import yaml

with open('config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
project_path = config['path']['colab']

!cp -r "{project_path}/utils"       /content/utils
!cp -r "{project_path}/data"        /content/data

Mounted at /content/drive


In [ ]:
# WandB 연결
import os, wandb
from google.colab import userdata
from utils.setting import wandb_start

os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
wandb.login()

wandb_run = wandb_start(config)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

from transformers import DataCollatorWithPadding, set_seed

import warnings
warnings.filterwarnings("ignore")

random_seed = config['seed']
set_seed(random_seed)

# 2. Data Preprocessing

In [ ]:
# from utils.dataset import ReviewDataset
# from utils.preprocessing import TextPreprocessingPipeline_Base, TextPreprocessingPipeline, Proofreading

# df = pd.read_csv("data/train.csv")
# print(f"기본 전처리 적용 전 데이터: {len(df)}개")

# # NULL, 빈 텍스트, 스페이스 제거
# df = df[~df["review"].isnull()]
# df = df[~df["review"].str.strip().eq("")]
# df = df[~df["review"].str.isspace()]

# # 같은 리뷰, 다른 라벨 처리: 가장 빈도수 높은 라벨을 채택
# # 1. 중복 리뷰 추출
# duplicates = df[df["review"].duplicated(keep=False)]

# # 2. 중복 리뷰에서 가장 빈도수 높은 라벨 추출
# most_frequent_label = duplicates.groupby(['review'])['label'].agg(lambda x: x.value_counts().index[0])
# most_frequent_label = most_frequent_label.to_frame()
# most_frequent_label = most_frequent_label.reset_index()

# # 3. df와 중복 리뷰를 'review'를 기준으로 병합해 중복 리뷰의 최빈 라벨을 기록
# merged_df = pd.merge(df, most_frequent_label, how='left', on='review')

# # 4. label_y의 NaN 값을 label_x 값으로 채워넣기. 다시 말해 label_y(최빈 라벨) 값을 우선 채택
# merged_df['label'] = merged_df['label_y'].combine_first(merged_df['label_x'])
# merged_df = merged_df.drop(columns=['label_y', 'label_x'])
# merged_df['label'] = merged_df['label'].astype(int)

# # 5. 중복 리뷰 제거
# merged_df = merged_df.drop_duplicates(subset="review")
# df = merged_df
# # df.head(10)

# print(f"기본 전처리 적용 후 데이터: {len(df)}개")

# 3. Data Loading

원본 데이터의 클래스 분포
- 클래스 0: 55849개 (40.9%)
- 클래스 1: 13314개 (9.8%)
- 클래스 2: 48198개 (35.3%)
- 클래스 3: 19120개 (14.0%)

In [ ]:
# from utils.tokenizer import build_custom_tokenizer

# tokenizer       = build_custom_tokenizer(config)
# max_seq_length  = config['tokenizer']['max_length']
# train_batch     = config['train']['batch_size']
# valid_batch     = config['valid']['batch_size']
# train_workers   = config['train']['num_workers']
# valid_workers   = config['valid']['num_workers']

# augment_data    = df[df['type']=='augment']
# augment_data    = augment_data[augment_data["label"].isin([1,3])]
# augment_data    = augment_data.drop("type", axis=1)

# df = df[df['type']=='original']

# df['review'] = df['review'].apply(Proofreading)

# X   = df["review"]
# y   = df["label"]
# ids = df["ID"]

# X_train, X_valid, y_train, y_valid, ids_train, ids_valid = train_test_split(
#     X, y, ids, test_size=0.1, random_state=random_seed, stratify=y
# )

# preprocessor    = TextPreprocessingPipeline()

# X_train_        = preprocessor.fit_transform(X_train.tolist(), y_train.tolist())
# X_valid_        = preprocessor.transform(X_valid.tolist())

# train_data      = pd.DataFrame({"ID": ids_train, "review": X_train_, "label": y_train}).reset_index(drop=True)
# valid_data      = pd.DataFrame({"ID": ids_valid, "review": X_valid_, "label": y_valid}).reset_index(drop=True)

# train_data      = pd.concat([augment_data, train_data])

# train_data      = train_data[~train_data["review"].str.strip().eq("")]   # 34개의 빈 텍스트 삭제
# valid_data      = valid_data[~valid_data["review"].str.strip().eq("")]   # 34개의 빈 텍스트 삭제

# train_dataset   = ReviewDataset(train_data["review"], train_data["label"], tokenizer, max_seq_length)
# valid_dataset   = ReviewDataset(valid_data["review"], valid_data["label"], tokenizer, max_seq_length)

# train_loader    = DataLoader(train_dataset,
#                              batch_size = train_batch,
#                              collate_fn = DataCollatorWithPadding(tokenizer=tokenizer),
#                              num_workers= train_workers,
#                              shuffle    = True)
# valid_loader    = DataLoader(valid_dataset,
#                              batch_size   = valid_batch,
#                              collate_fn   = DataCollatorWithPadding(tokenizer=tokenizer),
#                              num_workers= valid_workers,
#                              shuffle      = False)

# label0  = (train_data['label']==0).sum()/len(train_data) * 100
# label1  = (train_data['label']==1).sum()/len(train_data) * 100
# label2  = (train_data['label']==2).sum()/len(train_data) * 100
# label3  = (train_data['label']==3).sum()/len(train_data) * 100

# print("=== [BERT 훈련 및 검증 데이터셋] ===")
# print(f"훈련 데이터: {len(train_dataset):,}개")
# print(f"검증 데이터: {len(valid_dataset):,}개")
# print(f"훈련 배치: {len(train_loader):,}개")
# print(f"검증 배치: {len(valid_loader):,}개")
# print(f"클래스 분포: [0(강한 부정)]:{label0:.2f}%, [1(약한 부정)]:{label1:.2f}%, [2(약한 긍정)]:{label2:.2f}%, [3(강한 긍정)]:{label3:.2f}%")

In [ ]:
# save_file_path = "./new_train.csv"
# train_data.to_csv(save_file_path, index=False)

# save_file_path = "./new_valid.csv"
# valid_data.to_csv(save_file_path, index=False)

In [ ]:
# 전처리를 미리 해두고 저장해둔 데이터 가져와 쓰기

from utils.tokenizer import build_custom_tokenizer
from utils.dataset import ReviewDataset

tokenizer       = build_custom_tokenizer(config)
max_seq_length  = config['tokenizer']['max_length']
train_batch     = config['train']['batch_size']
valid_batch     = config['valid']['batch_size']
train_workers   = config['train']['num_workers']
valid_workers   = config['valid']['num_workers']

train_data      = pd.read_csv("data/new_train.csv")
valid_data      = pd.read_csv("data/new_valid.csv")

train_dataset   = ReviewDataset(train_data["review"], train_data["label"], tokenizer, max_seq_length)
valid_dataset   = ReviewDataset(valid_data["review"], valid_data["label"], tokenizer, max_seq_length)

train_loader    = DataLoader(train_dataset,
                             batch_size = train_batch,
                             collate_fn = DataCollatorWithPadding(tokenizer=tokenizer),
                             num_workers= train_workers,
                             shuffle    = True)
valid_loader    = DataLoader(valid_dataset,
                             batch_size   = valid_batch,
                             collate_fn   = DataCollatorWithPadding(tokenizer=tokenizer),
                             num_workers= valid_workers,
                             shuffle      = False)

label0  = (train_data['label']==0).sum()/len(train_data) * 100
label1  = (train_data['label']==1).sum()/len(train_data) * 100
label2  = (train_data['label']==2).sum()/len(train_data) * 100
label3  = (train_data['label']==3).sum()/len(train_data) * 100

print("=== [BERT 훈련 및 검증 데이터셋] ===")
print(f"훈련 데이터: {len(train_dataset):,}개")
print(f"검증 데이터: {len(valid_dataset):,}개")
print(f"훈련 배치: {len(train_loader):,}개")
print(f"검증 배치: {len(valid_loader):,}개")
print(f"클래스 분포: [0(강한 부정)]:{label0:.2f}%, [1(약한 부정)]:{label1:.2f}%, [2(약한 긍정)]:{label2:.2f}%, [3(강한 긍정)]:{label3:.2f}%")

new tokens: ['oo', 'ㅡㅅㅡ', 'ㅗ', '^^', '용두사미', '망작', '꿀잼', '노잼', '남주']
=== [BERT 훈련 및 검증 데이터셋] ===
훈련 데이터: 155,881개
검증 데이터: 13,648개
훈련 배치: 4,872개
검증 배치: 107개
클래스 분포: [0(강한 부정)]:32.24%, [1(약한 부정)]:16.41%, [2(약한 긍정)]:27.82%, [3(강한 긍정)]:23.53%


# 4. Model Training

In [ ]:
from utils.optimizer import get_optimizer, get_scheduler
from utils.loss import get_criterion
from utils.model import get_model
from utils.train import train

In [ ]:
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_config = config['model']
optim_config = config['optimizer']

model       = get_model(model_config, tokenizer)
criterion   = get_criterion(config, train_data, device)
optimizer   = get_optimizer(model, optim_config)
scheduler   = get_scheduler(config, optimizer, len(train_loader))
num_epochs  = config['train']['epoch']

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


모델 적용: klue/bert-base, num_classes=4, dropout=(0.1, 0.1, 0.3)
손실 함수 적용: Smooth CE, smoothing: 0.1
옵티마이저 설정: AdamW
스케줄러 적용: LinearLR, 1 에폭
스케줄러 적용: Cosine Annealing 주기 3 에폭


In [ ]:
train(model             = model,
      criterion         = criterion,
      optimizer         = optimizer,
      train_loader      = train_loader,
      valid_loader      = valid_loader,
      epochs            = num_epochs,
      device            = device,
      scheduler         = scheduler,
      accum_step        = 1,
      )

======== Epoch: 1 ========


Epoch 1/4:   0%|          | 2/4872 [00:01<40:09,  2.02it/s]  

	Train Loss: 1.3996


Epoch 1/4:  20%|██        | 976/4872 [01:34<06:12, 10.45it/s]

	Train Loss: 1.0642


...

	Train Loss: 0.4982


Epoch 4/4: 100%|██████████| 4872/4872 [07:50<00:00, 10.36it/s]


	Train Loss: 0.4984


Validation: 100%|██████████| 107/107 [00:23<00:00,  4.52it/s]

Epoch Summary(4/4): Train Loss: 0.4984 // Val Loss: 0.7374 // Val Acc: 81.40%


In [ ]:
wandb.finish()

# 5. Confusion Matrix

In [ ]:
from utils.utils import BERT_evaluation, compute_metrics, metrics_summary

In [ ]:
save_path = './best_model.pt'

finetuned_BERT = get_model(model_config, tokenizer)
finetuned_BERT.load_state_dict(torch.load(save_path, map_location='cuda'))
finetuned_BERT.eval()
print(f"파인 튜닝 모델 불러오기 완료")

pred_list, target_list = BERT_evaluation(finetuned_BERT, valid_loader, device)
metrics = compute_metrics(pred_list, target_list)
metrics_summary(metrics)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


모델 적용: klue/bert-base, num_classes=4, dropout=(0.1, 0.1, 0.3)
파인 튜닝 모델 불러오기 완료


Validation: 100%|██████████| 107/107 [00:05<00:00, 17.90it/s]

===== [Metrics Summary] =====
Valid Acc: 81.71%
F1_micro: 0.8171
F1_macro: 0.7547
F1_weighted: 0.8155
Confusion:
[[89.6  5.5  3.8  1.1]
 [24.5 51.7 23.   0.8]
 [ 4.9  4.5 83.7  6.9]
 [ 5.4  0.2 19.8 74.6]]


